In [ ]:
from collections import Counter
from pathlib import Path
import csv
import importlib.util
import sys

import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scripts' / '04_pilot_qa.py').exists():
    raise RuntimeError('Run this notebook with the repository root as the working directory.')

qa_path = PROJECT_ROOT / 'scripts' / '04_pilot_qa.py'
spec = importlib.util.spec_from_file_location('pilot_qa', qa_path)
pilot_qa = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = pilot_qa
spec.loader.exec_module(pilot_qa)

metadata_path = PROJECT_ROOT / 'data/raw/ucsd-google-local/meta-Vermont.json.gz'
reviews_path = PROJECT_ROOT / 'data/raw/ucsd-google-local/review-Vermont.json.gz'
places_path = PROJECT_ROOT / 'data/interim/government-places-Vermont.csv'
panel_path = PROJECT_ROOT / 'data/processed/place-month-Vermont.csv'
for path in (metadata_path, reviews_path, places_path, panel_path):
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Run make pilot-download and make pilot-all first.')
print('Inputs found; all paths are repository-relative.')

In [ ]:
metadata_profile = pilot_qa.inspect_raw_archive(metadata_path, 'metadata')
review_profile = pilot_qa.inspect_raw_archive(reviews_path, 'reviews')
archive_summary = {
    'metadata_rows': metadata_profile['rows'],
    'metadata_unique_places': len(metadata_profile['gmap_ids']),
    'metadata_exact_duplicate_excess': metadata_profile['exact_duplicate_rows'],
    'review_rows': review_profile['rows'],
    'review_unique_places': len(review_profile['gmap_ids']),
    'review_exact_duplicate_excess': review_profile['exact_duplicate_rows'],
    'review_fingerprint_duplicate_excess': review_profile['fingerprint_duplicate_rows'],
    'missing_ratings': review_profile['missing_rating'],
    'minimum_review_date': str(review_profile['minimum_time']),
    'maximum_review_date': str(review_profile['maximum_time']),
    'review_place_ids_missing_from_metadata': len(review_profile['gmap_ids'] - metadata_profile['gmap_ids']),
}
archive_summary

In [ ]:
with places_path.open(newline='', encoding='utf-8') as handle:
    candidate_rows = list(csv.DictReader(handle))
candidate_by_id = {}
for row in candidate_rows:
    candidate_by_id.setdefault(row['gmap_id'], row)
retrieval_channels = Counter(row['matched_fields'] for row in candidate_by_id.values())
address_only = sum(row['address_only_match'] == 'yes' for row in candidate_by_id.values())

with panel_path.open(newline='', encoding='utf-8') as handle:
    panel_rows = list(csv.DictReader(handle))
cell_counts = [int(row['n_reviews']) for row in panel_rows]
analysis_reviews = sum(cell_counts)
sparse_cells = sum(count < 5 for count in cell_counts)
result_summary = {
    'candidate_rows': len(candidate_rows),
    'unique_candidate_place_ids': len(candidate_by_id),
    'address_only_candidates': address_only,
    'observed_place_months': len(panel_rows),
    'deduplicated_analysis_reviews': analysis_reviews,
    'place_months_below_5_reviews': sparse_cells,
    'share_below_5_reviews': sparse_cells / len(panel_rows),
}
result_summary, retrieval_channels

In [ ]:
family_counts = Counter(row['service_family'] for row in candidate_by_id.values())
families, counts = zip(*sorted(family_counts.items(), key=lambda item: item[1]))
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(families, counts, color='#35618f')
ax.set_title('Rule-retrieved Vermont candidate listings by service family')
ax.set_xlabel('Unique candidate place listings')
ax.set_ylabel('Service family')
ax.text(0, -0.16, 'Retrieval candidates, not a validated government-facility universe.', transform=ax.transAxes, fontsize=9)
fig.tight_layout()
plt.show()